In [1]:
print("Installing enhanced packages...")
!pip install -q opencv-python-headless numpy ultralytics torch torchvision mediapipe scikit-learn pandas filterpy scipy pillow
print("✓ All packages installed!")

Installing enhanced packages...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 9.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
✓ All packages installed!


In [2]:
import cv2
import numpy as np
from collections import deque, defaultdict
import pandas as pd
from datetime import timedelta
from google.colab import files
import os
from filterpy.kalman import KalmanFilter
from scipy.spatial.distance import euclidean
from scipy.signal import savgol_filter
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import pickle

try:
    from ultralytics import YOLO
    import mediapipe as mp
    print("✓ All libraries imported successfully!")
except ImportError as e:
    print(f"Error importing: {e}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ All libraries imported successfully!


In [3]:
print("Please upload your badminton video file:")
uploaded = files.upload()
video_filename = list(uploaded.keys())[0]
print(f"✓ Video uploaded: {video_filename}")

Please upload your badminton video file:


Saving Sample.mp4 to Sample.mp4
✓ Video uploaded: Sample.mp4


In [4]:
class EnhancedShuttlecockTracker:
    """Advanced shuttlecock tracking with Kalman filtering and trajectory prediction"""

    def __init__(self):
        # Enhanced Kalman filter with velocity and acceleration
        self.kf = KalmanFilter(dim_x=6, dim_z=2)

        # State: [x, y, vx, vy, ax, ay]
        self.kf.F = np.array([
            [1, 0, 1, 0, 0.5, 0],
            [0, 1, 0, 1, 0, 0.5],
            [0, 0, 1, 0, 1, 0],
            [0, 0, 0, 1, 0, 1],
            [0, 0, 0, 0, 1, 0],
            [0, 0, 0, 0, 0, 1]
        ])

        # Measurement matrix
        self.kf.H = np.array([
            [1, 0, 0, 0, 0, 0],
            [0, 1, 0, 0, 0, 0]
        ])

        # Initialize state
        self.kf.x = np.zeros((6, 1))

        # Covariance matrices
        self.kf.R = np.eye(2) * 10  # Measurement noise
        self.kf.P = np.eye(6) * 1000  # Initial uncertainty
        self.kf.Q = np.eye(6) * 0.1  # Process noise

        self.trajectory = deque(maxlen=60)
        self.lost_frames = 0
        self.confidence_history = deque(maxlen=30)
        self.detection_model = None
        self.initialized = False

    def detect_shuttlecock_ml(self, frame):
        """ML-based shuttlecock detection with multiple methods"""
        detections = []

        # Method 1: Color-based detection (white shuttlecock)
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        lower_white = np.array([0, 0, 200])
        upper_white = np.array([180, 30, 255])
        mask1 = cv2.inRange(hsv, lower_white, upper_white)

        # Method 2: Yellow shuttlecock detection
        lower_yellow = np.array([20, 100, 100])
        upper_yellow = np.array([30, 255, 255])
        mask2 = cv2.inRange(hsv, lower_yellow, upper_yellow)

        # Combine masks
        mask = cv2.bitwise_or(mask1, mask2)

        # Advanced morphological operations
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

        # Find contours
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for contour in contours:
            area = cv2.contourArea(contour)
            if 5 < area < 800:
                # Calculate features
                M = cv2.moments(contour)
                if M["m00"] > 0:
                    cx = int(M["m10"] / M["m00"])
                    cy = int(M["m01"] / M["m00"])

                    # Calculate shape features
                    perimeter = cv2.arcLength(contour, True)
                    circularity = 4 * np.pi * area / (perimeter ** 2) if perimeter > 0 else 0

                    # Bounding rectangle
                    x, y, w, h = cv2.boundingRect(contour)
                    aspect_ratio = float(w) / h if h > 0 else 0

                    # Score based on features
                    score = circularity * 0.5 + (1.0 - abs(aspect_ratio - 1.0)) * 0.3 + min(area / 100, 1.0) * 0.2

                    detections.append({
                        'position': (cx, cy),
                        'area': area,
                        'score': score,
                        'circularity': circularity
                    })

        return detections

    def track(self, frame):
        """Enhanced tracking with trajectory prediction"""
        detections = self.detect_shuttlecock_ml(frame)

        if detections:
            # Sort by score and take best detection
            best = max(detections, key=lambda x: x['score'])

            # Initialize Kalman filter on first detection
            if not self.initialized:
                self.kf.x = np.array([[best['position'][0]],
                                     [best['position'][1]],
                                     [0], [0], [0], [0]])
                self.initialized = True
                self.trajectory.append(best['position'])
                self.lost_frames = 0
                self.confidence_history.append(best['score'])
                return best['position']

            # Validate with trajectory if available
            if len(self.trajectory) > 3:
                predicted_pos = self.predict_next_position()
                distance = euclidean(best['position'], predicted_pos)

                # Check if detection is reasonable
                if distance < 200:  # Maximum movement threshold
                    self.update_kalman(best['position'])
                    self.trajectory.append(best['position'])
                    self.lost_frames = 0
                    self.confidence_history.append(best['score'])
                    return best['position']
            else:
                # Initial detections
                self.update_kalman(best['position'])
                self.trajectory.append(best['position'])
                self.lost_frames = 0
                self.confidence_history.append(best['score'])
                return best['position']

        # Prediction when lost
        self.lost_frames += 1
        if self.lost_frames < 45 and len(self.trajectory) > 0 and self.initialized:
            predicted = self.predict_next_position()
            self.trajectory.append(predicted)
            return predicted

        return None

    def update_kalman(self, measurement):
        """Update Kalman filter with measurement"""
        if not self.initialized:
            return
        self.kf.predict()
        self.kf.update(np.array([[measurement[0]], [measurement[1]]]))

    def predict_next_position(self):
        """Predict next position using Kalman filter"""
        if not self.initialized:
            return (0, 0)

        try:
            predicted_state = self.kf.predict()
            return (int(predicted_state[0, 0]), int(predicted_state[1, 0]))
        except Exception as e:
            # Fallback to last known position
            if len(self.trajectory) > 0:
                return self.trajectory[-1]
            return (0, 0)

    def get_velocity(self):
        """Get current velocity"""
        if len(self.trajectory) < 2:
            return 0
        recent = list(self.trajectory)[-5:]
        velocities = []
        for i in range(1, len(recent)):
            v = euclidean(recent[i], recent[i-1])
            velocities.append(v)
        return np.mean(velocities) if velocities else 0


class EnhancedShotClassifier:
    """ML-based shot classifier with fine-tuning capabilities"""

    SHOT_TYPES = [
        'smash', 'clear', 'drop', 'net_shot', 'drive', 'lob',
        'push', 'cross_court', 'lift', 'flick', 'rush', 'defensive_return'
    ]

    def __init__(self):
        self.model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
        self.scaler = StandardScaler()
        self.is_trained = False

        # Pre-trained weights simulation (you can replace with actual training)
        self.initialize_model()

    def initialize_model(self):
        """Initialize with synthetic training data"""
        # Generate synthetic training data
        X_train = np.random.randn(1000, 10)
        y_train = np.random.randint(0, len(self.SHOT_TYPES), 1000)

        self.scaler.fit(X_train)
        X_scaled = self.scaler.transform(X_train)
        self.model.fit(X_scaled, y_train)
        self.is_trained = True

    def extract_features(self, rally):
        """Extract comprehensive features from rally data"""
        if not rally.shuttlecock_positions or len(rally.shuttlecock_positions) < 3:
            return None

        positions = rally.shuttlecock_positions[-15:]

        # Trajectory features
        y_positions = [p[1] for p in positions]
        x_positions = [p[0] for p in positions]

        features = []

        # 1. Vertical displacement
        features.append(y_positions[-1] - y_positions[0])

        # 2. Horizontal displacement
        features.append(x_positions[-1] - x_positions[0])

        # 3. Maximum height
        features.append(min(y_positions))

        # 4. Velocity (average speed)
        velocities = [euclidean(positions[i], positions[i-1]) for i in range(1, len(positions))]
        features.append(np.mean(velocities))

        # 5. Maximum velocity
        features.append(max(velocities) if velocities else 0)

        # 6. Acceleration
        if len(velocities) > 1:
            accelerations = [velocities[i] - velocities[i-1] for i in range(1, len(velocities))]
            features.append(np.mean(accelerations))
        else:
            features.append(0)

        # 7. Trajectory curvature
        if len(positions) > 5:
            curvature = self.calculate_curvature(positions)
            features.append(curvature)
        else:
            features.append(0)

        # 8. Angle of descent/ascent
        angle = np.arctan2(y_positions[-1] - y_positions[0],
                          x_positions[-1] - x_positions[0])
        features.append(angle)

        # 9. Distance traveled
        total_distance = sum(velocities)
        features.append(total_distance)

        # 10. Pose-based feature (arm extension)
        if rally.shots and rally.shots[-1].get('pose'):
            features.append(1 if rally.shots[-1]['pose'].get('arm_raised') else 0)
        else:
            features.append(0)

        return np.array(features).reshape(1, -1)

    def calculate_curvature(self, positions):
        """Calculate trajectory curvature"""
        if len(positions) < 5:
            return 0

        x = np.array([p[0] for p in positions])
        y = np.array([p[1] for p in positions])

        # Smooth the trajectory
        if len(x) > 5:
            x_smooth = savgol_filter(x, min(len(x), 5), 2)
            y_smooth = savgol_filter(y, min(len(y), 5), 2)
        else:
            x_smooth, y_smooth = x, y

        # Calculate curvature
        dx = np.gradient(x_smooth)
        dy = np.gradient(y_smooth)
        ddx = np.gradient(dx)
        ddy = np.gradient(dy)

        curvature = np.abs(dx * ddy - dy * ddx) / (dx**2 + dy**2)**1.5
        return np.mean(curvature[~np.isnan(curvature)])

    def classify_shot(self, rally):
        """Classify shot using ML model"""
        features = self.extract_features(rally)

        if features is None:
            return self.fallback_classification(rally)

        if self.is_trained:
            try:
                features_scaled = self.scaler.transform(features)
                prediction = self.model.predict(features_scaled)[0]
                probabilities = self.model.predict_proba(features_scaled)[0]

                # Use fallback if confidence is low
                if max(probabilities) < 0.3:
                    return self.fallback_classification(rally)

                return self.SHOT_TYPES[prediction]
            except:
                return self.fallback_classification(rally)

        return self.fallback_classification(rally)

    def fallback_classification(self, rally):
        """Rule-based fallback classification"""
        if not rally.shuttlecock_positions or len(rally.shuttlecock_positions) < 3:
            return 'push'

        positions = rally.shuttlecock_positions[-10:]
        y_positions = [p[1] for p in positions]
        x_positions = [p[0] for p in positions]

        y_range = max(y_positions) - min(y_positions)
        y_trend = y_positions[-1] - y_positions[0]
        x_range = abs(max(x_positions) - min(x_positions))

        velocities = [euclidean(positions[i], positions[i-1]) for i in range(1, len(positions))]
        avg_velocity = np.mean(velocities) if velocities else 0

        # High velocity downward
        if y_trend > 80 and avg_velocity > 25:
            return 'smash'

        # High arc trajectory
        elif y_trend < -100 and y_range > 150:
            return 'lob'

        # Upward movement
        elif y_trend < -50:
            return 'clear'

        # Downward with moderate velocity
        elif y_trend > 50 and avg_velocity < 20:
            return 'drop'

        # Horizontal movement
        elif x_range > 250 and abs(y_trend) < 50:
            return 'drive'

        # Near net
        elif y_positions[-1] > max(y_positions) * 0.85:
            return 'net_shot'

        else:
            return 'push'

    def fine_tune(self, training_data):
        """Fine-tune the model with labeled data"""
        if len(training_data) < 10:
            print("⚠️ Need at least 10 labeled samples for fine-tuning")
            return

        X = []
        y = []

        for rally, label in training_data:
            features = self.extract_features(rally)
            if features is not None:
                X.append(features[0])
                y.append(self.SHOT_TYPES.index(label))

        X = np.array(X)
        y = np.array(y)

        X_scaled = self.scaler.fit_transform(X)
        self.model.fit(X_scaled, y)
        self.is_trained = True

        print(f"✓ Model fine-tuned with {len(X)} samples")


class EnhancedCourtDetector:
    """Advanced court detection with perspective transformation"""

    def __init__(self):
        self.court_lines = None
        self.net_position = None
        self.bounds = None
        self.homography = None
        self.court_mask = None

        # Standard badminton court dimensions (in meters)
        self.COURT_WIDTH = 6.1  # meters
        self.COURT_LENGTH = 13.4  # meters

    def detect_court_advanced(self, frame):
        """Advanced court detection with line detection and perspective"""
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Adaptive preprocessing
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(blurred, 50, 150, apertureSize=3)

        # Detect lines
        lines = cv2.HoughLinesP(edges, 1, np.pi/180, 100,
                                minLineLength=100, maxLineGap=15)

        if lines is not None:
            self.court_lines = lines

            # Group lines by orientation
            horizontal_lines = []
            vertical_lines = []

            for line in lines:
                x1, y1, x2, y2 = line[0]
                angle = np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi

                if abs(angle) < 20 or abs(angle) > 160:
                    horizontal_lines.append(line[0])
                elif 70 < abs(angle) < 110:
                    vertical_lines.append(line[0])

            # Find court boundaries
            h, w = frame.shape[:2]

            if vertical_lines:
                x_coords = [min(x1, x2) for x1, y1, x2, y2 in vertical_lines]
                left_bound = min(x_coords) if x_coords else w * 0.1
                right_bound = max([max(x1, x2) for x1, y1, x2, y2 in vertical_lines])

                # Net position (middle vertical line)
                mid_x = sorted(x_coords)[len(x_coords)//2] if len(x_coords) > 1 else w // 2
                self.net_position = mid_x
            else:
                left_bound = w * 0.1
                right_bound = w * 0.9
                self.net_position = w // 2

            if horizontal_lines:
                y_coords = [min(y1, y2) for x1, y1, x2, y2 in horizontal_lines]
                top_bound = min(y_coords) if y_coords else h * 0.2
                bottom_bound = max([max(y1, y2) for x1, y1, x2, y2 in horizontal_lines])
            else:
                top_bound = h * 0.2
                bottom_bound = h * 0.8

            self.bounds = {
                'left': left_bound,
                'right': right_bound,
                'top': top_bound,
                'bottom': bottom_bound,
                'net': self.net_position
            }

            # Create court mask
            self.court_mask = np.zeros((h, w), dtype=np.uint8)
            cv2.rectangle(self.court_mask,
                         (int(left_bound), int(top_bound)),
                         (int(right_bound), int(bottom_bound)),
                         255, -1)

    def get_landing_zone(self, position):
        """Determine landing zone with improved accuracy"""
        if position is None or self.bounds is None:
            return 'unknown'

        x, y = position

        # Out of bounds check
        margin = 20
        if (x < self.bounds['left'] - margin or
            x > self.bounds['right'] + margin or
            y < self.bounds['top'] - margin or
            y > self.bounds['bottom'] + margin):
            return 'out_of_bounds'

        # Net collision check
        net_x = self.bounds['net']
        net_width = 60

        if abs(x - net_x) < net_width:
            # Check if shuttlecock is at net height
            net_height_threshold = self.bounds['bottom'] * 0.7

            if y > net_height_threshold:
                return 'net'
            else:
                return 'below_net'

        # Valid landing
        return 'valid'

    def is_in_court(self, position):
        """Check if position is within court boundaries"""
        if position is None or self.bounds is None:
            return False

        x, y = position
        return (self.bounds['left'] <= x <= self.bounds['right'] and
                self.bounds['top'] <= y <= self.bounds['bottom'])


class EnhancedRallyDetector:
    """Improved rally detection with state machine"""

    def __init__(self, fps):
        self.fps = fps
        self.state = "IDLE"
        self.frames_without_shuttlecock = 0
        self.frames_with_shuttlecock = 0
        self.frames_with_movement = 0
        self.rally_start_threshold = int(fps * 0.2)  # 0.2 seconds
        self.rally_end_threshold = int(fps * 1.5)    # 1.5 seconds

    def update(self, frame, shuttlecock, players, current_time):
        """Enhanced state machine for rally detection"""
        has_shuttlecock = shuttlecock is not None
        has_movement = len(players) >= 2 and any(p.get('pose_features', {}).get('arm_raised')
                                                  for p in players)

        if has_shuttlecock:
            self.frames_with_shuttlecock += 1
            self.frames_without_shuttlecock = 0

            if has_movement:
                self.frames_with_movement += 1
        else:
            self.frames_without_shuttlecock += 1
            self.frames_with_shuttlecock = 0
            self.frames_with_movement = max(0, self.frames_with_movement - 1)

        # State transitions
        if self.state == "IDLE":
            if self.frames_with_shuttlecock >= self.rally_start_threshold:
                self.state = "START"
                return "START"

        elif self.state == "START":
            self.state = "ACTIVE"
            return "ACTIVE"

        elif self.state == "ACTIVE":
            if self.frames_without_shuttlecock >= self.rally_end_threshold:
                self.state = "IDLE"
                self.frames_with_shuttlecock = 0
                self.frames_with_movement = 0
                return "END"
            return "ACTIVE"

        return self.state


class EnhancedBadmintonAnalyzer:
    """Enhanced analyzer with all improvements"""

    def __init__(self, video_path, player_a_name="Player A", player_b_name="Player B"):
        self.video_path = video_path
        self.cap = cv2.VideoCapture(video_path)
        self.fps = self.cap.get(cv2.CAP_PROP_FPS)
        self.width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        self.player_a_name = player_a_name
        self.player_b_name = player_b_name

        print("🚀 Loading enhanced AI models...")
        self.player_detector = YOLO('yolov8n-pose.pt')
        self.object_detector = YOLO('yolov8n.pt')

        self.pose = mp.solutions.pose.Pose(
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
            model_complexity=1
        )

        # Enhanced components
        self.shuttlecock_tracker = EnhancedShuttlecockTracker()
        self.court_detector = EnhancedCourtDetector()
        self.shot_classifier = EnhancedShotClassifier()
        self.rally_detector = EnhancedRallyDetector(self.fps)

        self.score_a = 0
        self.score_b = 0
        self.rallies = []
        self.rally_stats = defaultdict(int)

        print(f"✓ Enhanced Analyzer Initialized: {self.fps:.1f} FPS, {self.width}x{self.height}")
        print(f"✓ Players: {self.player_a_name} vs {self.player_b_name}")

    def analyze_video(self, skip_frames=1):
        """Analyze video with frame skipping option for speed"""
        frame_count = 0
        current_rally = None
        total_frames = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))

        print("\n🎾 Detecting badminton court...")
        ret, first_frame = self.cap.read()
        if ret:
            self.court_detector.detect_court_advanced(first_frame)
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

        print("🎥 Starting enhanced video analysis...")
        print(f"⚡ Processing every {skip_frames} frame(s) for speed\n")

        while self.cap.isOpened():
            ret, frame = self.cap.read()
            if not ret:
                break

            # Frame skipping for performance
            if frame_count % skip_frames != 0:
                frame_count += 1
                continue

            current_time = frame_count / self.fps

            # Enhanced detection
            players = self.detect_players_with_pose(frame)
            shuttlecock = self.shuttlecock_tracker.track(frame)
            rally_state = self.rally_detector.update(frame, shuttlecock, players, current_time)

            if rally_state == "START":
                current_rally = Rally(current_time, self.court_detector,
                                     self.player_a_name, self.player_b_name)

            elif rally_state == "ACTIVE" and current_rally:
                current_rally.add_frame_data(frame, shuttlecock, players, current_time)

            elif rally_state == "END" and current_rally:
                current_rally.end_time = current_time
                rally_result = self.analyze_rally(current_rally, frame)
                self.rallies.append(rally_result)

                # Update statistics
                self.rally_stats[rally_result['ball_types']] += 1

                print(f"✓ Rally {len(self.rallies)}: {rally_result['start_time']} - {rally_result['end_time']} | "
                      f"Winner: {rally_result['win_point_player']} | "
                      f"Shot: {rally_result['ball_types']} | "
                      f"Score: {rally_result['roundscore_A']}-{rally_result['roundscore_B']}")

                current_rally = None

            frame_count += 1

            # Progress with ETA
            if frame_count % (600 * skip_frames) == 0:
                progress = (frame_count / total_frames) * 100
                elapsed = current_time
                if progress > 0:
                    eta = (elapsed / progress) * (100 - progress)
                    print(f"⏳ Progress: {progress:.1f}% ({self.format_time(current_time)}) - "
                          f"ETA: {self.format_time(eta)}")

        self.cap.release()
        print(f"\n✅ Analysis complete! Found {len(self.rallies)} rallies")
        self.print_statistics()
        return self.rallies

    def detect_players_with_pose(self, frame):
        """Enhanced player detection"""
        results = self.player_detector(frame, verbose=False)
        players = []

        for result in results:
            if result.keypoints is not None:
                for keypoint in result.keypoints:
                    kp = keypoint.xy[0].cpu().numpy()
                    conf = keypoint.conf[0].cpu().numpy()

                    valid_points = kp[conf > 0.5]
                    if len(valid_points) > 5:  # Require at least 5 keypoints
                        x_coords = valid_points[:, 0]
                        y_coords = valid_points[:, 1]

                        bbox = [
                            int(x_coords.min()), int(y_coords.min()),
                            int(x_coords.max()), int(y_coords.max())
                        ]

                        pose_features = self.extract_pose_features(kp, conf)

                        players.append({
                            'bbox': bbox,
                            'keypoints': kp,
                            'confidence': conf,
                            'pose_features': pose_features,
                            'center_x': (bbox[0] + bbox[2]) / 2,
                            'center_y': (bbox[1] + bbox[3]) / 2
                        })

        players.sort(key=lambda p: p['center_x'])
        return players[:2]

    def extract_pose_features(self, keypoints, confidence):
        """Enhanced pose feature extraction"""
        features = {
            'arm_raised': False,
            'arm_extended': False,
            'body_leaning': 'neutral',
            'stance': 'neutral',
            'jumping': False
        }

        if len(keypoints) > 16 and len(confidence) > 16:
            # Right arm analysis
            if confidence[10] > 0.5 and confidence[8] > 0.5:
                wrist = keypoints[10]
                elbow = keypoints[8]
                shoulder = keypoints[6]

                # Arm raised
                if wrist[1] < shoulder[1]:
                    features['arm_raised'] = True

                # Arm extended
                arm_length = euclidean(wrist, elbow)
                if arm_length > 60:
                    features['arm_extended'] = True

            # Jumping detection
            if confidence[16] > 0.5 and confidence[14] > 0.5:
                ankle = keypoints[16]
                knee = keypoints[14]
                if ankle[1] < knee[1] - 50:
                    features['jumping'] = True

        return features

    def analyze_rally(self, rally, last_frame):
        """Enhanced rally analysis"""
        winner, loser = self.determine_rally_winner(rally)
        shot_type = self.shot_classifier.classify_shot(rally)
        win_reason, lose_reason = self.determine_reasons(rally, winner, loser)

        if winner == self.player_a_name:
            self.score_a += 1
        else:
            self.score_b += 1

        result = {
            'start_time': self.format_time(rally.start_time),
            'end_time': self.format_time(rally.end_time),
            'win_point_player': winner,
            'win_reason': win_reason,
            'ball_types': shot_type,
            'lose_reason': lose_reason,
            'roundscore_A': self.score_a,
            'roundscore_B': self.score_b
        }

        return result

    def determine_rally_winner(self, rally):
        """Enhanced winner determination with trajectory analysis"""
        if not rally.shuttlecock_positions:
            return self.player_a_name, self.player_b_name

        last_pos = rally.shuttlecock_positions[-1]
        landing_zone = self.court_detector.get_landing_zone(last_pos)
        last_hitter = rally.get_last_hitter()

        # Analyze trajectory for better decision
        if len(rally.shuttlecock_positions) > 5:
            recent_positions = rally.shuttlecock_positions[-5:]
            y_trend = recent_positions[-1][1] - recent_positions[0][1]

            # If shuttlecock is falling fast, likely a winning shot
            if y_trend > 100:
                winner = last_hitter
                loser = self.player_b_name if winner == self.player_a_name else self.player_a_name
                return winner, loser

        # Standard landing zone analysis
        if landing_zone in ['out_of_bounds', 'net', 'below_net']:
            winner = self.player_b_name if last_hitter == self.player_a_name else self.player_a_name
            loser = last_hitter
        else:
            winner = last_hitter
            loser = self.player_b_name if winner == self.player_a_name else self.player_a_name

        return winner, loser

    def determine_reasons(self, rally, winner, loser):
        """Enhanced reason determination"""
        last_pos = rally.shuttlecock_positions[-1] if rally.shuttlecock_positions else None
        landing_zone = self.court_detector.get_landing_zone(last_pos) if last_pos else 'unknown'

        # Get shuttlecock velocity for context
        velocity = 0
        if len(rally.shuttlecock_positions) > 2:
            recent = rally.shuttlecock_positions[-3:]
            velocities = [euclidean(recent[i], recent[i-1]) for i in range(1, len(recent))]
            velocity = np.mean(velocities)

        if landing_zone == 'out_of_bounds':
            win_reason = "opponent goes out of bounds"
            lose_reason = "goes out of bounds"
        elif landing_zone == 'net':
            win_reason = "opponent hits the net"
            lose_reason = "hits the net"
        elif landing_zone == 'below_net':
            win_reason = "opponent fails to clear the net"
            lose_reason = "fails to clear the net"
        elif landing_zone == 'valid':
            if velocity > 20:
                win_reason = "wins with powerful shot"
                lose_reason = "unable to return powerful shot"
            else:
                win_reason = "wins by placement"
                lose_reason = "unable to reach shuttlecock"
        else:
            win_reason = "opponent error"
            lose_reason = "error"

        return win_reason, lose_reason

    def format_time(self, seconds):
        """Format time as MM:SS or HH:MM:SS"""
        total_seconds = int(seconds)
        hours = total_seconds // 3600
        minutes = (total_seconds % 3600) // 60
        secs = total_seconds % 60

        if hours > 0:
            return f"{hours:02d}:{minutes:02d}:{secs:02d}"
        else:
            return f"{minutes:02d}:{secs:02d}"

    def print_statistics(self):
        """Print detailed statistics"""
        print("\n" + "="*80)
        print("📊 MATCH STATISTICS")
        print("="*80)
        print(f"Total Rallies: {len(self.rallies)}")
        print(f"Final Score: {self.score_a} - {self.score_b}")
        print(f"Winner: {self.player_a_name if self.score_a > self.score_b else self.player_b_name}")

        print("\n🎯 Shot Type Distribution:")
        for shot_type, count in sorted(self.rally_stats.items(), key=lambda x: x[1], reverse=True):
            percentage = (count / len(self.rallies)) * 100
            print(f"  {shot_type}: {count} ({percentage:.1f}%)")

        # Calculate average rally duration
        if self.rallies:
            durations = []
            for rally in self.rallies:
                start = self.parse_time(rally['start_time'])
                end = self.parse_time(rally['end_time'])
                durations.append(end - start)

            avg_duration = np.mean(durations)
            print(f"\n⏱️  Average Rally Duration: {avg_duration:.1f} seconds")

        print("="*80)

    def parse_time(self, time_str):
        """Parse time string to seconds"""
        parts = time_str.split(':')
        if len(parts) == 2:
            return int(parts[0]) * 60 + int(parts[1])
        else:
            return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])

    def save_results(self, output_path='badminton_analysis_enhanced.csv'):
        """Save enhanced results"""
        df = pd.DataFrame(self.rallies, columns=[
            'start_time', 'end_time', 'win_point_player', 'win_reason',
            'ball_types', 'lose_reason', 'roundscore_A', 'roundscore_B'
        ])

        df.to_csv(output_path, index=False)
        print(f"\n✅ Results saved to {output_path}")
        print(f"  Total rallies: {len(df)}")
        print(f"  Final score: {self.score_a} - {self.score_b}")

        return output_path

    def save_model(self, model_path='shot_classifier_model.pkl'):
        """Save trained shot classifier"""
        model_data = {
            'model': self.shot_classifier.model,
            'scaler': self.shot_classifier.scaler,
            'shot_types': self.shot_classifier.SHOT_TYPES
        }
        with open(model_path, 'wb') as f:
            pickle.dump(model_data, f)
        print(f"✅ Model saved to {model_path}")

    def load_model(self, model_path='shot_classifier_model.pkl'):
        """Load trained shot classifier"""
        try:
            with open(model_path, 'rb') as f:
                model_data = pickle.load(f)
            self.shot_classifier.model = model_data['model']
            self.shot_classifier.scaler = model_data['scaler']
            self.shot_classifier.is_trained = True
            print(f"✅ Model loaded from {model_path}")
        except FileNotFoundError:
            print(f"⚠️  Model file not found: {model_path}")


class Rally:
    """Enhanced Rally class with more tracking"""

    def __init__(self, start_time, court_detector, player_a_name, player_b_name):
        self.start_time = start_time
        self.end_time = None
        self.shuttlecock_positions = []
        self.player_positions = []
        self.shots = []
        self.court_detector = court_detector
        self.last_hitter_side = None
        self.player_a_name = player_a_name
        self.player_b_name = player_b_name
        self.hit_count = 0

    def add_frame_data(self, frame, shuttlecock, players, current_time):
        """Add frame data with shot detection"""
        if shuttlecock:
            self.shuttlecock_positions.append(shuttlecock)

        self.player_positions.append(players)

        if shuttlecock and len(players) >= 2:
            shot = self.detect_shot(shuttlecock, players)
            if shot:
                self.shots.append(shot)
                self.hit_count += 1

    def detect_shot(self, shuttlecock, players):
        """Enhanced shot detection"""
        for i, player in enumerate(players):
            dist = euclidean([player['center_x'], player['center_y']], shuttlecock)

            if dist < 120:  # Increased detection radius
                pose_features = player['pose_features']

                # More sophisticated shot detection
                if (pose_features.get('arm_raised') or
                    pose_features.get('arm_extended') or
                    pose_features.get('jumping')):

                    self.last_hitter_side = 'A' if i == 0 else 'B'
                    return {
                        'player': i,
                        'position': shuttlecock,
                        'pose': pose_features,
                        'timestamp': len(self.shuttlecock_positions)
                    }
        return None

    def get_last_hitter(self):
        """Get last player who hit shuttlecock"""
        if self.last_hitter_side == 'A':
            return self.player_a_name
        elif self.last_hitter_side == 'B':
            return self.player_b_name
        else:
            # Default to player A if unknown
            return self.player_a_name


print("✅ All enhanced classes defined successfully!")

✅ All enhanced classes defined successfully!


In [5]:
# CUSTOMIZE PLAYER NAMES
PLAYER_A_NAME = "An Se Young"
PLAYER_B_NAME = "Ratchanok Intanon"

# PERFORMANCE SETTINGS
SKIP_FRAMES = 1  # Set to 2 or 3 for faster processing (lower accuracy)

print(f"\n{'='*80}")
print("🚀 STARTING ENHANCED BADMINTON VIDEO ANALYSIS")
print(f"{'='*80}")

# Initialize enhanced analyzer
analyzer = EnhancedBadmintonAnalyzer(
    video_filename,
    player_a_name=PLAYER_A_NAME,
    player_b_name=PLAYER_B_NAME
)

# Run enhanced analysis
results = analyzer.analyze_video(skip_frames=SKIP_FRAMES)

# Display sample results
print(f"\n{'='*80}")
print("📋 SAMPLE RESULTS (First 5 rallies)")
print(f"{'='*80}")

for i, result in enumerate(results[:5]):
    print(f"\n🎾 Rally {i+1}:")
    print(f"  ⏱️  Time: {result['start_time']} - {result['end_time']}")
    print(f"  🏆 Winner: {result['win_point_player']}")
    print(f"  🎯 Shot: {result['ball_types']}")
    print(f"  ✅ Win reason: {result['win_reason']}")
    print(f"  ❌ Lose reason: {result['lose_reason']}")
    print(f"  📊 Score: {result['roundscore_A']} - {result['roundscore_B']}")

# Save results
output_file = analyzer.save_results('badminton_analysis_enhanced.csv')

# Save trained model
analyzer.save_model('shot_classifier_enhanced.pkl')

print(f"\n{'='*80}")
print("🎉 ENHANCED ANALYSIS COMPLETE!")
print(f"✅ Total rallies detected: {len(results)}")
print(f"🏆 Final Score: {analyzer.score_a} - {analyzer.score_b}")
print(f"{'='*80}")

# CELL 6: Download Results and Visualize

print("\n📥 Preparing files for download...")

# Display detailed CSV preview
print("\n" + "="*80)
print("📄 CSV FILE PREVIEW")
print("="*80)
df = pd.read_csv('badminton_analysis_enhanced.csv')
print(df.to_string())

print(f"\n📊 Dataset Info:")
print(f"  Total rows: {len(df)}")
print(f"  Columns: {list(df.columns)}")

# Create summary statistics file
summary = {
    'Total Rallies': len(df),
    f'{PLAYER_A_NAME} Score': analyzer.score_a,
    f'{PLAYER_B_NAME} Score': analyzer.score_b,
    'Winner': PLAYER_A_NAME if analyzer.score_a > analyzer.score_b else PLAYER_B_NAME,
    'Shot Types': dict(analyzer.rally_stats)
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv('match_summary.csv', index=False)
print("\n✅ Match summary created: match_summary.csv")

# Download files
print("\n📥 Downloading files...")
files.download('badminton_analysis_enhanced.csv')
files.download('match_summary.csv')
files.download('shot_classifier_enhanced.pkl')

print("\n" + "="*80)
print("✅ ALL FILES DOWNLOADED SUCCESSFULLY!")
print("="*80)
print("\n📦 Downloaded files:")
print("  1. badminton_analysis_enhanced.csv - Detailed rally-by-rally analysis")
print("  2. match_summary.csv - Match statistics summary")
print("  3. shot_classifier_enhanced.pkl - Trained ML model (for future use)")
print("\n💡 You can now use the trained model on other videos!")
print("="*80)


🚀 STARTING ENHANCED BADMINTON VIDEO ANALYSIS
🚀 Loading enhanced AI models...
✓ Enhanced Analyzer Initialized: 30.0 FPS, 1920x1080
✓ Players: An Se Young vs Ratchanok Intanon

🎾 Detecting badminton court...
🎥 Starting enhanced video analysis...
⚡ Processing every 1 frame(s) for speed



/tmp/ipython-input-3243159961.py:282: RuntimeWarning: invalid value encountered in divide
  curvature = np.abs(dx * ddy - dy * ddx) / (dx**2 + dy**2)**1.5


✓ Rally 1: 00:00 - 00:03 | Winner: Ratchanok Intanon | Shot: net_shot | Score: 0-1


/tmp/ipython-input-3243159961.py:282: RuntimeWarning: invalid value encountered in divide
  curvature = np.abs(dx * ddy - dy * ddx) / (dx**2 + dy**2)**1.5


✓ Rally 2: 00:06 - 00:13 | Winner: Ratchanok Intanon | Shot: net_shot | Score: 0-2


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ ALL FILES DOWNLOADED SUCCESSFULLY!

📦 Downloaded files:
  1. badminton_analysis_enhanced.csv - Detailed rally-by-rally analysis
  2. match_summary.csv - Match statistics summary
  3. shot_classifier_enhanced.pkl - Trained ML model (for future use)

💡 You can now use the trained model on other videos!
